In [2]:
"""
=============================================================================
NCAA March Mania 2026 — FINAL SUBMISSION GENERATOR
=============================================================================
Produces a single submission_stage2.csv with exactly 132,133 rows.
Men   (TeamID 1xxx) : trained on M files, men's best BO params
Women (TeamID 3xxx) : trained on W files, women's best BO params

Steps:
  1  Load data
  2  Compute ELO (2010–2025)
  3  Build team stats, recent form, Massey (men only), seeds
  4  Build training matchup features
  5  Train LightGBM + XGBoost, tune ensemble weight
  6  Build 2026 prediction features (2025 stats + end-of-2025 ELO)
  7  Predict all rows → clip to [0.025, 0.975]
  8  Save + sanity check
=============================================================================
"""

import warnings; warnings.filterwarnings("ignore")
import os, multiprocessing, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss
import lightgbm as lgb
import xgboost as xgb

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
DATA_DIR = Path("/Users/shaurya/Downloads")

N_JOBS = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"]      = str(N_JOBS)
os.environ["MKL_NUM_THREADS"]      = str(N_JOBS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_JOBS)
print(f"🖥️  Using {N_JOBS} CPU cores")

TRAIN_SEASONS = list(range(2010, 2026))   # 2010–2025 inclusive
ELO_K         = 20
ELO_BASE      = 1500
ELO_HOME_ADV  = 100
SEED          = 42
np.random.seed(SEED)

STAT_COLS = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
             "OR","DR","Ast","TO","Stl","Blk","PF"]

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — MEN'S (from screenshots)
# ─────────────────────────────────────────────────────────────
MEN_LGBM = {
    "num_leaves"        : 170,
    "max_depth"         : 12,
    "learning_rate"     : 0.005,
    "n_estimators"      : 1308 + 200,
    "min_child_samples" : 5,
    "subsample"         : 0.5,
    "colsample_bytree"  : 0.5,
    "reg_alpha"         : 0.0,
    "reg_lambda"        : 0.0,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "device_type"       : "cpu",
    "num_threads"       : N_JOBS,
    "verbose"           : -1,
}
MEN_XGB = {
    "n_estimators"          : 1638 + 200,
    "max_depth"             : 10,
    "learning_rate"         : 0.0255778,
    "min_child_weight"      : 2.84511,
    "subsample"             : 0.848826,
    "colsample_bytree"      : 0.562814,
    "gamma"                 : 1.59748,
    "reg_alpha"             : 4.86392,
    "reg_lambda"            : 1.04622,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — WOMEN'S (from screenshots)
# ─────────────────────────────────────────────────────────────
WOMEN_LGBM = {
    "num_leaves"        : 164,
    "max_depth"         : 8,
    "learning_rate"     : 0.0187029,
    "n_estimators"      : 1294 + 200,
    "min_child_samples" : 22,
    "subsample"         : 0.532526,
    "colsample_bytree"  : 0.974443,
    "reg_alpha"         : 4.82816,
    "reg_lambda"        : 4.04199,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "device_type"       : "cpu",
    "num_threads"       : N_JOBS,
    "verbose"           : -1,
}
WOMEN_XGB = {
    "n_estimators"          : 1898 + 200,
    "max_depth"             : 7,
    "learning_rate"         : 0.0559337,
    "min_child_weight"      : 18.9736,
    "subsample"             : 0.842968,
    "colsample_bytree"      : 0.895046,
    "gamma"                 : 0.171643,
    "reg_alpha"             : 1.13418,
    "reg_lambda"            : 2.44057,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}

# ─────────────────────────────────────────────────────────────
# SHARED HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────
def parse_seed_num(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16

def compute_elo(reg_df, tourney_df):
    """ELO with MoV multiplier, home advantage, season carry-over."""
    cols = ["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]
    all_games = pd.concat([reg_df[cols], tourney_df[cols]]) \
                  .sort_values(["Season","DayNum"]).reset_index(drop=True)
    elo = {}
    season_end = {}
    for season in sorted(all_games["Season"].unique()):
        for tid in list(elo):
            elo[tid] = ELO_BASE * 0.25 + elo[tid] * 0.75
        for _, r in all_games[all_games["Season"] == season].iterrows():
            w, l   = int(r["WTeamID"]), int(r["LTeamID"])
            rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
            loc    = r["WLoc"]
            rw_adj = rw + ELO_HOME_ADV if loc=="H" \
                     else (rw - ELO_HOME_ADV if loc=="A" else rw)
            mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
            mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
            exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
            delta  = ELO_K * mov_m * (1 - exp_w)
            elo[w] = rw + delta
            elo[l] = rl - delta
        for tid, rat in elo.items():
            season_end[(season, tid)] = rat
    return season_end, elo   # elo = end-of-last-season ratings

def explode_games(games):
    rw = {"WTeamID":"TeamID","LTeamID":"OppID",
          "WScore":"PtsFor","LScore":"PtsAgainst",
          **{f"W{s}":s for s in STAT_COLS},
          **{f"L{s}":f"Opp{s}" for s in STAT_COLS}}
    rl = {"LTeamID":"TeamID","WTeamID":"OppID",
          "LScore":"PtsFor","WScore":"PtsAgainst",
          **{f"L{s}":s for s in STAT_COLS},
          **{f"W{s}":f"Opp{s}" for s in STAT_COLS}}
    keep = ["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst"] + \
           STAT_COLS + [f"Opp{s}" for s in STAT_COLS]
    w = games.rename(columns=rw); w["Win"] = 1
    l = games.rename(columns=rl); l["Win"] = 0
    return pd.concat([w[keep+["Win"]], l[keep+["Win"]]], ignore_index=True)

def build_team_stats(reg_df, tourney_df):
    tg  = explode_games(pd.concat([reg_df, tourney_df]))
    eps = 1e-6
    agg = tg.groupby(["Season","TeamID"]).agg(
        GP=("Win","count"), Wins=("Win","sum"),
        PtsFor=("PtsFor","mean"), PtsAgainst=("PtsAgainst","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"),   DR=("DR","mean"),
        Ast=("Ast","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"), PF=("PF","mean"),
        OppFGM=("OppFGM","mean"),   OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
        OppOR=("OppOR","mean"),     OppDR=("OppDR","mean"),
    ).reset_index()
    agg["WinPct"]    = agg["Wins"]    / (agg["GP"]      + eps)
    agg["FGPct"]     = agg["FGM"]     / (agg["FGA"]     + eps)
    agg["FG3Pct"]    = agg["FGM3"]    / (agg["FGA3"]    + eps)
    agg["FTPct"]     = agg["FTM"]     / (agg["FTA"]     + eps)
    agg["OppFGPct"]  = agg["OppFGM"]  / (agg["OppFGA"]  + eps)
    agg["eFGPct"]    = (agg["FGM"]    + 0.5*agg["FGM3"])    / (agg["FGA"]    + eps)
    agg["OppEFGPct"] = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / (agg["OppFGA"] + eps)
    poss             = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    agg["TOVRate"]   = agg["TO"]  / (poss              + eps)
    agg["ORBRate"]   = agg["OR"]  / (agg["OR"] + agg["OppDR"] + eps)
    agg["FTRate"]    = agg["FTM"] / (agg["FGA"]        + eps)
    agg["DRBRate"]   = agg["DR"]  / (agg["DR"] + agg["OppOR"] + eps)
    agg["Pace"]      = poss       / (agg["GP"]          + eps)
    agg["NetRtg"]    = agg["PtsFor"] - agg["PtsAgainst"]
    agg["OffRtg"]    = agg["PtsFor"]     / (poss + eps) * 100
    agg["DefRtg"]    = agg["PtsAgainst"] / (poss + eps) * 100
    agg["NetRtg2"]   = agg["OffRtg"] - agg["DefRtg"]
    agg["AstTO"]     = agg["Ast"] / (agg["TO"]  + eps)
    agg["StlBlk"]    = (agg["Stl"] + agg["Blk"]) / (agg["OppFGA"] + eps)
    agg["ThreePtRate"]    = agg["FGA3"]    / (agg["FGA"]    + eps)
    agg["OppThreePtRate"] = agg["OppFGA3"] / (agg["OppFGA"] + eps)
    return agg

def build_recent_form(reg_df, n=10):
    tg  = explode_games(reg_df).sort_values(["Season","TeamID","DayNum"])
    rec = tg.groupby(["Season","TeamID"]).tail(n)
    rf  = rec.groupby(["Season","TeamID"]).agg(
        RecentWinPct=("Win","mean"),
        RecentPtsFor=("PtsFor","mean"),
        RecentPtsAgainst=("PtsAgainst","mean"),
    ).reset_index()
    rf["RecentNetPts"] = rf["RecentPtsFor"] - rf["RecentPtsAgainst"]
    return rf

def build_massey(seasons):
    fp = DATA_DIR / "MMasseyOrdinals.csv"
    if not fp.exists(): return None
    df = pd.read_csv(fp)
    df = df[(df["Season"].isin(seasons)) & (df["RankingDayNum"] <= 128)]
    result = (df.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":"AvgRank"}))
    for sys in ["SAG","POM","WOL","MOR","DOK","REW","NET"]:
        sub = df[df["SystemName"] == sys]
        if len(sub) == 0: continue
        s = (sub.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":f"Rank_{sys}"}))
        result = result.merge(s, on=["Season","TeamID"], how="left")
    return result

def make_matchup_features(games, team_stats, recent, seed_feat,
                           massey_feat, elo_dict):
    df = games.copy()
    df["TeamA"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["TeamA"]).astype(int)

    def mg(df, stats, prefix, side):
        cols = [c for c in stats.columns if c not in ["Season","TeamID"]]
        s    = stats.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(s, left_on=["Season", side],
                        right_on=["Season","TeamID"], how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    rf = recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                 "RecentPtsFor","RecentPtsAgainst"]]
    df = mg(df, team_stats, "A", "TeamA")
    df = mg(df, team_stats, "B", "TeamB")
    df = mg(df, rf,         "A", "TeamA")
    df = mg(df, rf,         "B", "TeamB")
    df = mg(df, seed_feat,  "A", "TeamA")
    df = mg(df, seed_feat,  "B", "TeamB")
    if massey_feat is not None:
        mc = [c for c in massey_feat.columns if c not in ["Season","TeamID"]]
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "A", "TeamA")
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "B", "TeamB")

    # Prior season end-of-season ELO
    df["A_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamA"])), ELO_BASE), axis=1)
    df["B_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamB"])), ELO_BASE), axis=1)

    feature_cols = []
    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]
            feature_cols.extend([f"diff_{base}", ca, cb])
    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    feature_cols = list(dict.fromkeys(feature_cols + ["ELO_WinProb"]))
    return df, feature_cols

def build_pred_features(pairs_df, stats_2025, recent_2025,
                         seed_2025, massey_2025, elo_final, feature_cols):
    """Build prediction features for 2026 matchups using 2025 stats."""
    df = pairs_df.copy()

    def mg(df, ref, prefix, key):
        cols = [c for c in ref.columns if c != "TeamID"]
        renamed = ref.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(renamed, left_on=key, right_on="TeamID", how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    s  = stats_2025.drop(columns=["Season"])
    r  = recent_2025[["TeamID","RecentWinPct","RecentNetPts",
                       "RecentPtsFor","RecentPtsAgainst"]]
    sd = seed_2025.drop(columns=["Season"])

    df = mg(df, s,  "A", "TeamA")
    df = mg(df, s,  "B", "TeamB")
    df = mg(df, r,  "A", "TeamA")
    df = mg(df, r,  "B", "TeamB")
    df = mg(df, sd, "A", "TeamA")
    df = mg(df, sd, "B", "TeamB")

    if massey_2025 is not None and len(massey_2025) > 0:
        mc = massey_2025.drop(columns=["Season"])
        df = mg(df, mc, "A", "TeamA")
        df = mg(df, mc, "B", "TeamB")

    # End-of-2025 ELO
    df["A_ELO"] = df["TeamA"].apply(lambda t: elo_final.get(int(t), ELO_BASE))
    df["B_ELO"] = df["TeamB"].apply(lambda t: elo_final.get(int(t), ELO_BASE))

    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    return df.reindex(columns=feature_cols).fillna(0)

def train_and_predict(reg_df, tourney_df, seeds_df, massey_feat,
                      lgbm_params, xgb_params, pred_pairs, label):
    """Full pipeline for one gender."""

    print(f"\n{'█'*60}")
    print(f"  {label} PIPELINE")
    print(f"{'█'*60}")

    # ── STEP 2: ELO ─────────────────────────────────────────
    print(f"\n  STEP 2 | Computing ELO …")
    elo_dict, elo_final = compute_elo(reg_df, tourney_df)
    print(f"          {len(elo_dict):,} (season, team) ELO entries")

    # ── STEP 3: Features ─────────────────────────────────────
    print(f"  STEP 3 | Building team stats, recent form, seeds …")
    team_stats = build_team_stats(reg_df, tourney_df)
    recent     = build_recent_form(reg_df)

    seed_feat = seeds_df[seeds_df["Season"].isin(TRAIN_SEASONS)].copy()
    seed_feat["SeedNum"] = seed_feat["Seed"].apply(parse_seed_num)
    seed_feat = seed_feat[["Season","TeamID","SeedNum"]]

    print(f"          Team-season rows : {len(team_stats):,}")
    print(f"          Massey           : {'available' if massey_feat is not None else 'N/A (women)'}")

    # ── STEP 4: Matchup features ──────────────────────────────
    print(f"  STEP 4 | Building matchup feature matrix …")
    reg_df["IsTourney"]     = False
    tourney_df["IsTourney"] = True
    all_train = pd.concat([reg_df, tourney_df], ignore_index=True)

    train_df, feature_cols = make_matchup_features(
        all_train, team_stats, recent, seed_feat, massey_feat, elo_dict
    )
    X_tr = train_df[feature_cols].fillna(0)
    y_tr = train_df["Label"]
    print(f"          Train rows : {len(X_tr):,}  |  Features : {len(feature_cols)}")

    # ── STEP 5: Train models ──────────────────────────────────
    print(f"  STEP 5 | Training LightGBM …")
    lgbm_m = lgb.LGBMClassifier(**lgbm_params)
    lgbm_m.fit(X_tr, y_tr)
    print(f"          ✓ LightGBM done")

    # XGBoost needs val split for early stopping
    split  = int(len(X_tr) * 0.9)
    X_trx  = X_tr.iloc[:split];  y_trx = y_tr.iloc[:split]
    X_valx = X_tr.iloc[split:];  y_valx = y_tr.iloc[split:]

    print(f"          Training XGBoost …")
    xgb_m = xgb.XGBClassifier(**xgb_params)
    xgb_m.fit(X_trx, y_trx, eval_set=[(X_valx, y_valx)], verbose=False)
    print(f"          ✓ XGBoost done")

    # Tune ensemble weight on val split
    p_lv = lgbm_m.predict_proba(X_valx)[:,1]
    p_xv = xgb_m.predict_proba(X_valx)[:,1]
    best_w, best_ll = 0.5, 99
    for w in np.arange(0.05, 0.96, 0.05):
        ll = log_loss(y_valx, w*p_lv + (1-w)*p_xv)
        if ll < best_ll:
            best_ll, best_w = ll, w
    print(f"          Optimal blend → LightGBM: {best_w:.2f}  XGBoost: {1-best_w:.2f}  "
          f"(val LogLoss={best_ll:.5f})")

    # ── STEP 6: Build 2026 prediction features ────────────────
    print(f"  STEP 6 | Building 2026 prediction features …")
    stats_2025  = team_stats[team_stats["Season"] == 2025].copy()
    recent_2025 = recent[recent["Season"] == 2025].copy()
    seed_2025   = seed_feat[seed_feat["Season"] == 2025].copy()
    massey_2025 = massey_feat[massey_feat["Season"] == 2025].copy() \
                  if massey_feat is not None else None

    X_pred = build_pred_features(
        pred_pairs, stats_2025, recent_2025,
        seed_2025, massey_2025, elo_final, feature_cols
    )
    print(f"          Prediction rows : {len(X_pred):,}")

    # ── STEP 7: Predict ───────────────────────────────────────
    print(f"  STEP 7 | Predicting …")
    p_lgbm = lgbm_m.predict_proba(X_pred)[:,1]
    p_xgb  = xgb_m.predict_proba(X_pred)[:,1]
    p_ens  = np.clip(best_w*p_lgbm + (1-best_w)*p_xgb, 0.025, 0.975)
    print(f"          ✓ Done  "
          f"(mean={p_ens.mean():.4f}  "
          f"min={p_ens.min():.4f}  "
          f"max={p_ens.max():.4f})")
    return p_ens


# ─────────────────────────────────────────────────────────────
# STEP 1 — LOAD STAGE 2 SAMPLE & SPLIT BY GENDER
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 1 | Loading Stage 2 sample submission …")
print("="*60)

stage2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")
parts           = stage2["ID"].str.split("_", expand=True)
stage2["Season"] = parts[0].astype(int)
stage2["TeamA"]  = parts[1].astype(int)
stage2["TeamB"]  = parts[2].astype(int)

# Men: TeamID 1xxx  |  Women: TeamID 3xxx
men_mask   = stage2["TeamA"] < 3000
women_mask = stage2["TeamA"] >= 3000
men_pairs   = stage2[men_mask][["TeamA","TeamB"]].copy()
women_pairs = stage2[women_mask][["TeamA","TeamB"]].copy()

print(f"  Total rows   : {len(stage2):,}")
print(f"  Men's rows   : {len(men_pairs):,}  (TeamID 1xxx)")
print(f"  Women's rows : {len(women_pairs):,}  (TeamID 3xxx)")

# ── Load men's raw data ───────────────────────────────────────
print("\n  Loading Men's raw data …")
m_reg     = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
m_seeds   = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_massey  = build_massey(TRAIN_SEASONS)

m_reg     = m_reg[m_reg["Season"].isin(TRAIN_SEASONS)].copy()
m_tourney = m_tourney[m_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  M Regular: {len(m_reg):,}  |  M Tourney: {len(m_tourney):,}  |  "
      f"Massey: {'yes' if m_massey is not None else 'no'}")

# ── Load women's raw data ─────────────────────────────────────
print("  Loading Women's raw data …")
w_reg     = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")
w_seeds   = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

w_reg     = w_reg[w_reg["Season"].isin(TRAIN_SEASONS)].copy()
w_tourney = w_tourney[w_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  W Regular: {len(w_reg):,}  |  W Tourney: {len(w_tourney):,}  |  Massey: N/A")

# ─────────────────────────────────────────────────────────────
# RUN MEN'S PIPELINE
# ─────────────────────────────────────────────────────────────
men_preds = train_and_predict(
    m_reg, m_tourney, m_seeds, m_massey,
    MEN_LGBM, MEN_XGB,
    men_pairs,
    "🏀 MEN'S"
)

# ─────────────────────────────────────────────────────────────
# RUN WOMEN'S PIPELINE
# ─────────────────────────────────────────────────────────────
women_preds = train_and_predict(
    w_reg, w_tourney, w_seeds, None,   # No Massey for women
    WOMEN_LGBM, WOMEN_XGB,
    women_pairs,
    "🏀 WOMEN'S"
)

# ─────────────────────────────────────────────────────────────
# STEP 8 — COMBINE, SAVE, SANITY CHECK
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 8 | Combining and saving submission …")
print("="*60)

men_out             = stage2[men_mask][["ID"]].copy()
men_out["Pred"]     = men_preds.round(6)

women_out           = stage2[women_mask][["ID"]].copy()
women_out["Pred"]   = women_preds.round(6)

# Merge back in original sample order
submission = stage2[["ID"]].merge(
    pd.concat([men_out, women_out], ignore_index=True),
    on="ID", how="left"
)

# Final clip and round
submission["Pred"] = submission["Pred"].clip(0.025, 0.975).round(6)

# Save
out_path = DATA_DIR / "submission_stage2.csv"
submission[["ID","Pred"]].to_csv(out_path, index=False)

# ── Sanity check ─────────────────────────────────────────────
print(f"\n  {'─'*50}")
print(f"  File         : {out_path}")
print(f"  Total rows   : {len(submission):,}  (expected 132,133)")
print(f"  Men's rows   : {men_mask.sum():,}")
print(f"  Women's rows : {women_mask.sum():,}")
print(f"  Any NaN      : {submission['Pred'].isna().sum()}")
print(f"  Pred min     : {submission['Pred'].min():.4f}  (floor = 0.025)")
print(f"  Pred max     : {submission['Pred'].max():.4f}  (cap   = 0.975)")
print(f"  Pred mean    : {submission['Pred'].mean():.4f}  (should be ~0.5)")
print(f"  {'─'*50}")
print(f"\n  First 5 rows:")
print(submission.head(5).to_string(index=False))
print(f"\n  Last 5 rows:")
print(submission.tail(5).to_string(index=False))
print(f"\n✅  submission_stage2.csv is ready — upload directly to Kaggle!")

🖥️  Using 8 CPU cores

  STEP 1 | Loading Stage 2 sample submission …
  Total rows   : 132,133
  Men's rows   : 66,430  (TeamID 1xxx)
  Women's rows : 65,703  (TeamID 3xxx)

  Loading Men's raw data …
  M Regular: 84,808  |  M Tourney: 1,001  |  Massey: yes
  Loading Women's raw data …
  W Regular: 81,708  |  W Tourney: 961  |  Massey: N/A

████████████████████████████████████████████████████████████
  🏀 MEN'S PIPELINE
████████████████████████████████████████████████████████████

  STEP 2 | Computing ELO …
          5,690 (season, team) ELO entries
  STEP 3 | Building team stats, recent form, seeds …
          Team-season rows : 5,639
          Massey           : available
  STEP 4 | Building matchup feature matrix …
          Train rows : 85,809  |  Features : 172
  STEP 5 | Training LightGBM …
          ✓ LightGBM done
          Training XGBoost …
          ✓ XGBoost done
          Optimal blend → LightGBM: 0.95  XGBoost: 0.05  (val LogLoss=0.40268)
  STEP 6 | Building 2026 predictio

In [1]:
"""
=============================================================================
NCAA March Mania 2026 — FINAL SUBMISSION GENERATOR (FULLY FIXED)
=============================================================================
Produces submission_stage2.csv with exactly 132,133 rows.
Men   (TeamID 1xxx) : LightGBM + XGBoost ensemble, men's best BO params
Women (TeamID 3xxx) : LightGBM + XGBoost ensemble, women's best BO params

FIXES APPLIED vs previous version:
  ① 2026 seeds used for prediction (not 2025 seeds)
  ② Conference strength adjustment penalises weak-conference teams
  ③ Seed overrides re-added — 1 vs 16 can never be < 97.5% for 1-seed
  ④ ID-based prediction mapping (no reindex drift)
  ⑤ Prediction features use end-of-2025 ELO (not prior season)

=============================================================================
"""

import warnings; warnings.filterwarnings("ignore")
import os, multiprocessing, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
import lightgbm as lgb
import xgboost as xgb

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
DATA_DIR = Path("/Users/shaurya/Downloads")   # ← adjust if needed
OUTPUT   = DATA_DIR / "submission_stage2.csv"

N_JOBS = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"]      = str(N_JOBS)
os.environ["MKL_NUM_THREADS"]      = str(N_JOBS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_JOBS)
print(f"🖥️  Using {N_JOBS} CPU cores")

TRAIN_SEASONS = list(range(2010, 2026))   # 2010–2025 inclusive
ELO_K         = 20
ELO_BASE      = 1500
ELO_HOME_ADV  = 100
SEED          = 42
np.random.seed(SEED)

STAT_COLS = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
             "OR","DR","Ast","TO","Stl","Blk","PF"]

# ─────────────────────────────────────────────────────────────
# FIX ③ — SEED OVERRIDES (re-added)
# Forces extreme seed matchups to realistic probabilities.
# The model's raw output is overridden when it's clearly wrong.
# ─────────────────────────────────────────────────────────────
SEED_OVERRIDES_MENS = {
    # (lower_seed, higher_seed) : min probability for lower seed
    (1, 16): 0.975,   # 1-seed beats 16-seed at least 97.5% of time
    (1, 15): 0.945,
    (1, 14): 0.920,
    (1, 13): 0.900,
    (2, 15): 0.930,
    (2, 16): 0.960,
    (3, 14): 0.850,
    (4, 13): 0.820,
    (5, 12): 0.650,
    (6, 11): 0.620,
    (7, 10): 0.600,
    (8,  9): 0.520,
}
SEED_OVERRIDES_WOMENS = {
    (1, 16): 0.985,
    (1, 15): 0.965,
    (1, 14): 0.945,
    (1, 13): 0.925,
    (2, 15): 0.955,
    (2, 16): 0.975,
    (3, 14): 0.875,
    (4, 13): 0.850,
    (5, 12): 0.720,
    (6, 11): 0.680,
    (7, 10): 0.650,
    (8,  9): 0.540,
}

# ─────────────────────────────────────────────────────────────
# FIX ② — CONFERENCE STRENGTH TIERS
# Weak-conference teams have inflated stats vs weak opponents.
# We apply a multiplier to their WinPct, ELO, and ratings.
# ─────────────────────────────────────────────────────────────
# Power conferences — no adjustment
POWER_CONFS = {
    "acc","big_east","big_ten","big_twelve","pac_twelve",
    "sec","aac","mwc","wcc","a_ten"
}
# Mid-major conferences — slight downward adjustment
MID_MAJOR_CONFS = {
    "mvc","wac","sun_belt","cusa","mac","horizon",
    "big_sky","big_south","big_west","ovc","maac",
    "summit","southern","southland","caa","ivy",
    "patriot","nec","america_east","nec","a_sun"
}
# Weak conferences — stronger downward adjustment
WEAK_CONFS = {
    "meac","swac","nec","big_south","independent"
}

CONF_ELO_PENALTY = {
    "power"    :    0,   # no penalty
    "mid_major": -50,    # -50 ELO points
    "weak"     : -150,   # -150 ELO points (Howard plays in MEAC)
}

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — MEN'S (Bayesian Optimised)
# ─────────────────────────────────────────────────────────────
MEN_LGBM = {
    "num_leaves"        : 170,
    "max_depth"         : 12,
    "learning_rate"     : 0.005,
    "n_estimators"      : 1308 + 200,
    "min_child_samples" : 5,
    "subsample"         : 0.5,
    "colsample_bytree"  : 0.5,
    "reg_alpha"         : 0.0,
    "reg_lambda"        : 0.0,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "verbose"           : -1,
}
MEN_XGB = {
    "n_estimators"          : 1638 + 200,
    "max_depth"             : 10,
    "learning_rate"         : 0.0255778,
    "min_child_weight"      : 2.84511,
    "subsample"             : 0.848826,
    "colsample_bytree"      : 0.562814,
    "gamma"                 : 1.59748,
    "reg_alpha"             : 4.86392,
    "reg_lambda"            : 1.04622,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — WOMEN'S (Bayesian Optimised)
# ─────────────────────────────────────────────────────────────
WOMEN_LGBM = {
    "num_leaves"        : 164,
    "max_depth"         : 8,
    "learning_rate"     : 0.0187029,
    "n_estimators"      : 1294 + 200,
    "min_child_samples" : 22,
    "subsample"         : 0.532526,
    "colsample_bytree"  : 0.974443,
    "reg_alpha"         : 4.82816,
    "reg_lambda"        : 4.04199,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "verbose"           : -1,
}
WOMEN_XGB = {
    "n_estimators"          : 1898 + 200,
    "max_depth"             : 7,
    "learning_rate"         : 0.0559337,
    "min_child_weight"      : 18.9736,
    "subsample"             : 0.842968,
    "colsample_bytree"      : 0.895046,
    "gamma"                 : 0.171643,
    "reg_alpha"             : 1.13418,
    "reg_lambda"            : 2.44057,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}


# ═════════════════════════════════════════════════════════════
# SHARED HELPER FUNCTIONS
# ═════════════════════════════════════════════════════════════

def parse_seed_num(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16


def compute_elo(reg_df, tourney_df):
    """ELO with MoV multiplier, home advantage, season carry-over."""
    cols = ["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]
    all_games = pd.concat([reg_df[cols], tourney_df[cols]]) \
                  .sort_values(["Season","DayNum"]).reset_index(drop=True)
    elo = {}
    season_end = {}
    for season in sorted(all_games["Season"].unique()):
        for tid in list(elo):
            elo[tid] = ELO_BASE * 0.25 + elo[tid] * 0.75
        for _, r in all_games[all_games["Season"] == season].iterrows():
            w, l   = int(r["WTeamID"]), int(r["LTeamID"])
            rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
            loc    = r["WLoc"]
            rw_adj = rw + ELO_HOME_ADV if loc == "H" \
                     else (rw - ELO_HOME_ADV if loc == "A" else rw)
            mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
            mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
            exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
            delta  = ELO_K * mov_m * (1 - exp_w)
            elo[w] = rw + delta
            elo[l] = rl - delta
        for tid, rat in elo.items():
            season_end[(season, tid)] = rat
    return season_end, elo


def explode_games(games):
    rw = {"WTeamID":"TeamID","LTeamID":"OppID",
          "WScore":"PtsFor","LScore":"PtsAgainst",
          **{f"W{s}":s for s in STAT_COLS},
          **{f"L{s}":f"Opp{s}" for s in STAT_COLS}}
    rl = {"LTeamID":"TeamID","WTeamID":"OppID",
          "LScore":"PtsFor","WScore":"PtsAgainst",
          **{f"L{s}":s for s in STAT_COLS},
          **{f"W{s}":f"Opp{s}" for s in STAT_COLS}}
    keep = ["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst"] + \
           STAT_COLS + [f"Opp{s}" for s in STAT_COLS]
    w = games.rename(columns=rw); w["Win"] = 1
    l = games.rename(columns=rl); l["Win"] = 0
    return pd.concat([w[keep+["Win"]], l[keep+["Win"]]], ignore_index=True)


def build_team_stats(reg_df, tourney_df):
    tg  = explode_games(pd.concat([reg_df, tourney_df]))
    eps = 1e-6
    agg = tg.groupby(["Season","TeamID"]).agg(
        GP=("Win","count"),         Wins=("Win","sum"),
        PtsFor=("PtsFor","mean"),   PtsAgainst=("PtsAgainst","mean"),
        FGM=("FGM","mean"),         FGA=("FGA","mean"),
        FGM3=("FGM3","mean"),       FGA3=("FGA3","mean"),
        FTM=("FTM","mean"),         FTA=("FTA","mean"),
        OR=("OR","mean"),           DR=("DR","mean"),
        Ast=("Ast","mean"),         TO=("TO","mean"),
        Stl=("Stl","mean"),         Blk=("Blk","mean"),
        PF=("PF","mean"),
        OppFGM=("OppFGM","mean"),   OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
        OppOR=("OppOR","mean"),     OppDR=("OppDR","mean"),
    ).reset_index()
    agg["WinPct"]         = agg["Wins"]    / (agg["GP"]       + eps)
    agg["FGPct"]          = agg["FGM"]     / (agg["FGA"]      + eps)
    agg["FG3Pct"]         = agg["FGM3"]    / (agg["FGA3"]     + eps)
    agg["FTPct"]          = agg["FTM"]     / (agg["FTA"]      + eps)
    agg["OppFGPct"]       = agg["OppFGM"]  / (agg["OppFGA"]   + eps)
    agg["eFGPct"]         = (agg["FGM"]    + 0.5*agg["FGM3"])    / (agg["FGA"]    + eps)
    agg["OppEFGPct"]      = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / (agg["OppFGA"] + eps)
    poss                  = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    agg["TOVRate"]        = agg["TO"]  / (poss                   + eps)
    agg["ORBRate"]        = agg["OR"]  / (agg["OR"] + agg["OppDR"] + eps)
    agg["FTRate"]         = agg["FTM"] / (agg["FGA"]             + eps)
    agg["DRBRate"]        = agg["DR"]  / (agg["DR"] + agg["OppOR"] + eps)
    agg["Pace"]           = poss       / (agg["GP"]              + eps)
    agg["NetRtg"]         = agg["PtsFor"]     - agg["PtsAgainst"]
    agg["OffRtg"]         = agg["PtsFor"]     / (poss + eps) * 100
    agg["DefRtg"]         = agg["PtsAgainst"] / (poss + eps) * 100
    agg["NetRtg2"]        = agg["OffRtg"]     - agg["DefRtg"]
    agg["AstTO"]          = agg["Ast"]  / (agg["TO"]             + eps)
    agg["StlBlk"]         = (agg["Stl"] + agg["Blk"]) / (agg["OppFGA"] + eps)
    agg["ThreePtRate"]    = agg["FGA3"]    / (agg["FGA"]    + eps)
    agg["OppThreePtRate"] = agg["OppFGA3"] / (agg["OppFGA"] + eps)
    return agg


def build_recent_form(reg_df, n=10):
    tg  = explode_games(reg_df).sort_values(["Season","TeamID","DayNum"])
    rec = tg.groupby(["Season","TeamID"]).tail(n)
    rf  = rec.groupby(["Season","TeamID"]).agg(
        RecentWinPct=("Win","mean"),
        RecentPtsFor=("PtsFor","mean"),
        RecentPtsAgainst=("PtsAgainst","mean"),
    ).reset_index()
    rf["RecentNetPts"] = rf["RecentPtsFor"] - rf["RecentPtsAgainst"]
    return rf


def build_massey(seasons):
    fp = DATA_DIR / "MMasseyOrdinals.csv"
    if not fp.exists():
        print("  ⚠️  MMasseyOrdinals.csv not found — skipping Massey features")
        return None
    df = pd.read_csv(fp)
    df = df[(df["Season"].isin(seasons)) & (df["RankingDayNum"] <= 128)]
    result = (df.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":"AvgRank"}))
    for sys in ["SAG","POM","WOL","MOR","DOK","REW","NET"]:
        sub = df[df["SystemName"] == sys]
        if len(sub) == 0: continue
        s = (sub.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":f"Rank_{sys}"}))
        result = result.merge(s, on=["Season","TeamID"], how="left")
    return result


# ─────────────────────────────────────────────────────────────
# FIX ② — Conference strength adjustment
# ─────────────────────────────────────────────────────────────
def build_conf_penalty(conf_csv, seasons):
    """
    Returns dict {(season, teamID): elo_penalty}
    Power conf → 0, Mid-major → -50, Weak → -150
    """
    fp = DATA_DIR / conf_csv
    if not fp.exists():
        print(f"  ⚠️  {conf_csv} not found — skipping conference adjustment")
        return {}
    df = pd.read_csv(fp)
    df = df[df["Season"].isin(seasons)]
    penalty = {}
    for _, r in df.iterrows():
        conf = str(r["ConfAbbrev"]).lower()
        if conf in POWER_CONFS:
            p = CONF_ELO_PENALTY["power"]
        elif conf in WEAK_CONFS:
            p = CONF_ELO_PENALTY["weak"]
        else:
            p = CONF_ELO_PENALTY["mid_major"]
        penalty[(int(r["Season"]), int(r["TeamID"]))] = p
    return penalty


def apply_conf_penalty(elo_final, conf_penalty, season=2025):
    """Apply conference strength penalty to end-of-season ELO."""
    adjusted = {}
    for tid, rating in elo_final.items():
        pen = conf_penalty.get((season, tid), CONF_ELO_PENALTY["mid_major"])
        adjusted[tid] = rating + pen
    return adjusted


# ─────────────────────────────────────────────────────────────
# FIX ③ — Seed override application
# ─────────────────────────────────────────────────────────────
def apply_seed_overrides(preds, ids, seed_map_2026, seed_overrides):
    """
    For each prediction, check if the matchup has a seed override.
    If so, enforce the minimum probability for the favoured seed.
    seed_map_2026: {teamID: seed_number}
    """
    adj = preds.copy()
    parts = [i.split("_") for i in ids]

    for idx, (_, t1s, t2s) in enumerate(parts):
        t1, t2 = int(t1s), int(t2s)
        s1 = seed_map_2026.get(t1, 8)
        s2 = seed_map_2026.get(t2, 8)
        lo_seed, hi_seed = min(s1, s2), max(s1, s2)
        key = (lo_seed, hi_seed)

        if key in seed_overrides:
            ov = seed_overrides[key]
            # pred = P(t1 wins), t1 is always lower ID
            # figure out which team has the lower (better) seed
            if s1 < s2:
                # t1 is better seed → pred should be >= ov
                adj[idx] = max(adj[idx], ov)
            elif s2 < s1:
                # t2 is better seed → pred (P t1 wins) should be <= 1-ov
                adj[idx] = min(adj[idx], 1 - ov)
            # if seeds equal, no override

    n_overridden = int((np.abs(adj - preds) > 1e-6).sum())
    print(f"          Seed overrides applied : {n_overridden} predictions adjusted")
    return adj


def make_matchup_features(games, team_stats, recent, seed_feat,
                           massey_feat, elo_dict):
    """Build training matchup features — uses prior-season ELO."""
    df = games.copy()
    df["TeamA"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["TeamA"]).astype(int)

    def mg(df, stats, prefix, side):
        cols = [c for c in stats.columns if c not in ["Season","TeamID"]]
        s    = stats.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(s, left_on=["Season", side],
                        right_on=["Season","TeamID"], how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    rf = recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                 "RecentPtsFor","RecentPtsAgainst"]]
    df = mg(df, team_stats, "A", "TeamA")
    df = mg(df, team_stats, "B", "TeamB")
    df = mg(df, rf,         "A", "TeamA")
    df = mg(df, rf,         "B", "TeamB")
    df = mg(df, seed_feat,  "A", "TeamA")
    df = mg(df, seed_feat,  "B", "TeamB")
    if massey_feat is not None:
        mc = [c for c in massey_feat.columns if c not in ["Season","TeamID"]]
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "A", "TeamA")
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "B", "TeamB")

    df["A_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamA"])), ELO_BASE), axis=1)
    df["B_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamB"])), ELO_BASE), axis=1)

    feature_cols = []
    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]
            feature_cols.extend([f"diff_{base}", ca, cb])
    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    feature_cols = list(dict.fromkeys(feature_cols + ["ELO_WinProb"]))
    return df, feature_cols


def build_pred_features(pairs_df, stats_2025, recent_2025,
                         seed_2026,       # FIX ① — 2026 seeds, not 2025
                         massey_2025,
                         elo_final_adj,   # FIX ② — conf-adjusted ELO
                         feature_cols):
    """
    Build prediction features for 2026 matchups.
    FIX ① : uses 2026 tournament seeds (not 2025)
    FIX ② : uses conference-strength-adjusted end-of-2025 ELO
    """
    df = pairs_df.copy()

    def mg(df, ref, prefix, key):
        cols = [c for c in ref.columns if c != "TeamID"]
        renamed = ref.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(renamed, left_on=key, right_on="TeamID", how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    s  = stats_2025.drop(columns=["Season"])
    r  = recent_2025[["TeamID","RecentWinPct","RecentNetPts",
                       "RecentPtsFor","RecentPtsAgainst"]]
    sd = seed_2026   # ← FIX ① : 2026 seeds

    df = mg(df, s,  "A", "TeamA")
    df = mg(df, s,  "B", "TeamB")
    df = mg(df, r,  "A", "TeamA")
    df = mg(df, r,  "B", "TeamB")
    df = mg(df, sd, "A", "TeamA")
    df = mg(df, sd, "B", "TeamB")

    if massey_2025 is not None and len(massey_2025) > 0:
        mc = massey_2025.drop(columns=["Season"])
        df = mg(df, mc, "A", "TeamA")
        df = mg(df, mc, "B", "TeamB")

    # FIX ② : conference-adjusted ELO
    df["A_ELO"] = df["TeamA"].apply(lambda t: elo_final_adj.get(int(t), ELO_BASE))
    df["B_ELO"] = df["TeamB"].apply(lambda t: elo_final_adj.get(int(t), ELO_BASE))

    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    return df.reindex(columns=feature_cols).fillna(0)


# ═════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ═════════════════════════════════════════════════════════════

def run_pipeline(reg_df, tourney_df, seeds_df, massey_feat,
                 conf_csv, lgbm_params, xgb_params,
                 pred_pairs, seed_overrides, label):
    """
    Full training + prediction pipeline for one gender.
    Returns dict {submission_ID -> prediction}
    """
    print(f"\n{'█'*62}")
    print(f"  {label}")
    print(f"{'█'*62}")

    # ── STEP 2: ELO ──────────────────────────────────────────
    print(f"\n  STEP 2 | Computing ELO …")
    elo_dict, elo_final = compute_elo(reg_df, tourney_df)
    print(f"          {len(elo_dict):,} (season,team) ELO entries")
    print(f"          {len(elo_final):,} teams with end-of-2025 ELO")

    # FIX ② — apply conference strength penalty to prediction ELO
    print(f"  FIX ② | Applying conference strength penalty …")
    conf_penalty  = build_conf_penalty(conf_csv, TRAIN_SEASONS)
    elo_final_adj = apply_conf_penalty(elo_final, conf_penalty, season=2025)
    n_penalised   = sum(1 for v in conf_penalty.values() if v < 0)
    print(f"          {n_penalised:,} team-seasons penalised")

    # ── STEP 3: Features ─────────────────────────────────────
    print(f"  STEP 3 | Building team stats, recent form, seeds …")
    team_stats = build_team_stats(reg_df, tourney_df)
    recent     = build_recent_form(reg_df)

    # Training seeds (all historical seasons)
    seed_feat = seeds_df[seeds_df["Season"].isin(TRAIN_SEASONS)].copy()
    seed_feat["SeedNum"] = seed_feat["Seed"].apply(parse_seed_num)
    seed_feat = seed_feat[["Season","TeamID","SeedNum"]]
    print(f"          Team-season rows : {len(team_stats):,}")
    print(f"          Massey           : {'available' if massey_feat is not None else 'N/A'}")

    # ── STEP 4: Matchup feature matrix ───────────────────────
    print(f"  STEP 4 | Building matchup feature matrix …")
    reg_c     = reg_df.copy();     reg_c["IsTourney"]     = False
    tourney_c = tourney_df.copy(); tourney_c["IsTourney"] = True
    all_train = pd.concat([reg_c, tourney_c], ignore_index=True)

    train_df, feature_cols = make_matchup_features(
        all_train, team_stats, recent, seed_feat, massey_feat, elo_dict)
    X_tr = train_df[feature_cols].fillna(0)
    y_tr = train_df["Label"]
    print(f"          Train rows : {len(X_tr):,}  |  Features : {len(feature_cols)}")

    # Held-out 2025 tourney eval
    mask_eval = (train_df["Season"] == 2025) & (train_df["IsTourney"] == True)
    X_eval = train_df.loc[mask_eval, feature_cols].fillna(0) if mask_eval.sum() > 0 else None
    y_eval = train_df.loc[mask_eval, "Label"] if mask_eval.sum() > 0 else None

    # ── STEP 5: Train LightGBM ────────────────────────────────
    print(f"\n  STEP 5 | Training LightGBM on {len(X_tr):,} rows …")
    lgbm_m = lgb.LGBMClassifier(**lgbm_params)
    lgbm_m.fit(X_tr, y_tr)
    print(f"          ✓ LightGBM done")

    # XGBoost — needs val split for early stopping
    split  = int(len(X_tr) * 0.9)
    X_trx  = X_tr.iloc[:split];  y_trx = y_tr.iloc[:split]
    X_valx = X_tr.iloc[split:];  y_valx = y_tr.iloc[split:]
    print(f"          Training XGBoost …")
    xgb_m = xgb.XGBClassifier(**xgb_params)
    xgb_m.fit(X_trx, y_trx, eval_set=[(X_valx, y_valx)], verbose=False)
    print(f"          ✓ XGBoost done")

    # Tune ensemble blend weight on val split
    p_lv = lgbm_m.predict_proba(X_valx)[:,1]
    p_xv = xgb_m.predict_proba(X_valx)[:,1]
    best_w, best_ll = 0.5, 99
    for w in np.arange(0.05, 0.96, 0.05):
        ll = log_loss(y_valx, w*p_lv + (1-w)*p_xv)
        if ll < best_ll:
            best_ll, best_w = ll, w
    print(f"          Blend → LGBM: {best_w:.2f}  XGB: {1-best_w:.2f}  "
          f"val LogLoss={best_ll:.5f}")

    # 2025 tourney eval
    if X_eval is not None:
        p_l = lgbm_m.predict_proba(X_eval)[:,1]
        p_x = xgb_m.predict_proba(X_eval)[:,1]
        p_e = best_w*p_l + (1-best_w)*p_x
        acc = accuracy_score(y_eval, (p_e > 0.5).astype(int))
        nc  = int(((p_e > 0.5) == y_eval.values).sum())
        print(f"          [2025 Tourney] Accuracy={acc:.4f} ({nc}/{len(y_eval)})  "
              f"LogLoss={log_loss(y_eval, p_e):.5f}  "
              f"AUC={roc_auc_score(y_eval, p_e):.4f}")

    # ── STEP 6: 2026 prediction features ─────────────────────
    print(f"\n  STEP 6 | Building 2026 prediction features …")
    stats_2025  = team_stats[team_stats["Season"] == 2025].copy()
    recent_2025 = recent[recent["Season"] == 2025].copy()
    massey_2025 = massey_feat[massey_feat["Season"] == 2025].copy() \
                  if massey_feat is not None else None

    # FIX ① — use 2026 seeds, fall back to 2025 if 2026 not available
    if 2026 in seeds_df["Season"].values:
        seed_pred = seeds_df[seeds_df["Season"] == 2026].copy()
        print(f"          ✅ FIX ① : Using 2026 tournament seeds")
    else:
        seed_pred = seeds_df[seeds_df["Season"] == 2025].copy()
        print(f"          ⚠️  FIX ① : 2026 seeds not found, falling back to 2025 seeds")
    seed_pred["SeedNum"] = seed_pred["Seed"].apply(parse_seed_num)
    seed_pred = seed_pred[["TeamID","SeedNum"]]

    # Build seed map for override lookup {teamID: seed_number}
    seed_map_2026 = dict(zip(seed_pred["TeamID"], seed_pred["SeedNum"]))
    print(f"          {len(seed_map_2026)} teams have 2026 seed assignments")
    print(f"          Stats 2025  : {len(stats_2025):,} teams")
    print(f"          Recent 2025 : {len(recent_2025):,} teams")
    print(f"          Pred pairs  : {len(pred_pairs):,}")

    X_pred = build_pred_features(
        pred_pairs, stats_2025, recent_2025,
        seed_pred, massey_2025, elo_final_adj, feature_cols)
    print(f"          Feature matrix : {X_pred.shape}")

    # ── STEP 7: Predict ───────────────────────────────────────
    print(f"\n  STEP 7 | Predicting …")
    p_lgbm = lgbm_m.predict_proba(X_pred)[:,1]
    p_xgb  = xgb_m.predict_proba(X_pred)[:,1]
    preds  = best_w*p_lgbm + (1-best_w)*p_xgb

    # FIX ③ — apply seed overrides before clipping
    print(f"  FIX ③ | Applying seed overrides …")
    preds = apply_seed_overrides(
        preds, pred_pairs["ID"].values, seed_map_2026, seed_overrides)

    preds = np.clip(preds, 0.025, 0.975)
    print(f"          mean={preds.mean():.4f}  min={preds.min():.4f}  "
          f"max={preds.max():.4f}  std={preds.std():.4f}")

    # FIX ④ — ID dict mapping (no reindex drift)
    id_to_pred = dict(zip(pred_pairs["ID"].values, preds))
    print(f"          ✅ {label} done — {len(id_to_pred):,} predictions")
    return id_to_pred


# ═════════════════════════════════════════════════════════════
# STEP 1 — LOAD STAGE 2 SAMPLE & SPLIT BY GENDER
# ═════════════════════════════════════════════════════════════
print("\n" + "="*62)
print("  STEP 1 | Loading Stage 2 sample submission …")
print("="*62)

stage2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")
parts           = stage2["ID"].str.split("_", expand=True)
stage2["Season"] = parts[0].astype(int)
stage2["TeamA"]  = parts[1].astype(int)
stage2["TeamB"]  = parts[2].astype(int)

men_mask   = stage2["TeamA"] < 3000
women_mask = stage2["TeamA"] >= 3000
men_pairs   = stage2[men_mask][["ID","TeamA","TeamB"]].copy()
women_pairs = stage2[women_mask][["ID","TeamA","TeamB"]].copy()

print(f"  Total rows   : {len(stage2):,}")
print(f"  Men's rows   : {len(men_pairs):,}")
print(f"  Women's rows : {len(women_pairs):,}")

# ── Load men's data ───────────────────────────────────────────
print("\n  Loading Men's data …")
m_reg     = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
m_seeds   = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_massey  = build_massey(TRAIN_SEASONS)
m_reg     = m_reg[m_reg["Season"].isin(TRAIN_SEASONS)].copy()
m_tourney = m_tourney[m_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  M Regular: {len(m_reg):,}  M Tourney: {len(m_tourney):,}  "
      f"Massey: {'yes' if m_massey is not None else 'no'}")

# ── Load women's data ─────────────────────────────────────────
print("  Loading Women's data …")
w_reg     = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")
w_seeds   = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
w_reg     = w_reg[w_reg["Season"].isin(TRAIN_SEASONS)].copy()
w_tourney = w_tourney[w_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  W Regular: {len(w_reg):,}  W Tourney: {len(w_tourney):,}  Massey: N/A")


# ═════════════════════════════════════════════════════════════
# RUN BOTH PIPELINES
# ═════════════════════════════════════════════════════════════
mens_preds = run_pipeline(
    reg_df         = m_reg,
    tourney_df     = m_tourney,
    seeds_df       = m_seeds,
    massey_feat    = m_massey,
    conf_csv       = "MTeamConferences.csv",
    lgbm_params    = MEN_LGBM,
    xgb_params     = MEN_XGB,
    pred_pairs     = men_pairs,
    seed_overrides = SEED_OVERRIDES_MENS,
    label          = "🏀 MEN'S  (LGBM + XGBoost ensemble)",
)

womens_preds = run_pipeline(
    reg_df         = w_reg,
    tourney_df     = w_tourney,
    seeds_df       = w_seeds,
    massey_feat    = None,
    conf_csv       = "WTeamConferences.csv",
    lgbm_params    = WOMEN_LGBM,
    xgb_params     = WOMEN_XGB,
    pred_pairs     = women_pairs,
    seed_overrides = SEED_OVERRIDES_WOMENS,
    label          = "🏀 WOMEN'S  (LGBM + XGBoost ensemble)",
)


# ═════════════════════════════════════════════════════════════
# STEP 8 — COMBINE, SAVE, SANITY CHECK
# ═════════════════════════════════════════════════════════════
print("\n" + "="*62)
print("  STEP 8 | Combining and saving …")
print("="*62)

# FIX ④ — map by ID dict (zero reindex drift)
all_preds      = {**mens_preds, **womens_preds}
stage2["Pred"] = stage2["ID"].map(all_preds)

n_missing = stage2["Pred"].isna().sum()
if n_missing:
    print(f"  ⚠️  {n_missing} IDs unmatched — filling with 0.5")
    stage2["Pred"] = stage2["Pred"].fillna(0.5)

stage2["Pred"] = stage2["Pred"].clip(0.025, 0.975).round(6)
submission = stage2[["ID","Pred"]]
submission.to_csv(OUTPUT, index=False)

# ── Sanity check ─────────────────────────────────────────────
print(f"\n  {'─'*52}")
print(f"  File         : {OUTPUT}")
print(f"  Total rows   : {len(submission):,}  (expected 132,133) "
      f"{'✅' if len(submission)==132133 else '❌'}")
print(f"  Men's rows   : {men_mask.sum():,}")
print(f"  Women's rows : {women_mask.sum():,}")
print(f"  Any NaN      : {submission['Pred'].isna().sum()} "
      f"{'✅' if submission['Pred'].isna().sum()==0 else '❌'}")
print(f"  Pred min     : {submission['Pred'].min():.4f}  (floor=0.025)")
print(f"  Pred max     : {submission['Pred'].max():.4f}  (cap=0.975)")
print(f"  Pred mean    : {submission['Pred'].mean():.4f}  (should be ~0.5)")
print(f"  Pred std     : {submission['Pred'].std():.4f}  (>0.15 = good spread)")
print(f"  Unique vals  : {submission['Pred'].nunique():,}")
print(f"  {'─'*52}")
print(f"\n  First 5 rows:")
print(submission.head(5).to_string(index=False))
print(f"\n  Last 5 rows (women's):")
print(submission.tail(5).to_string(index=False))
print(f"\n✅  submission_stage2.csv ready — upload to Kaggle!")
print(f"\n  FIXES APPLIED:")
print(f"  ① 2026 seeds used for prediction features")
print(f"  ② Conference strength penalty on ELO (MEAC/SWAC = -150)")
print(f"  ③ Seed overrides: 1 vs 16 → 97.5% min for 1-seed")
print(f"  ④ ID dict mapping — no reindex drift")

🖥️  Using 8 CPU cores

  STEP 1 | Loading Stage 2 sample submission …
  Total rows   : 132,133
  Men's rows   : 66,430
  Women's rows : 65,703

  Loading Men's data …
  M Regular: 84,808  M Tourney: 1,001  Massey: yes
  Loading Women's data …
  W Regular: 81,708  W Tourney: 961  Massey: N/A

██████████████████████████████████████████████████████████████
  🏀 MEN'S  (LGBM + XGBoost ensemble)
██████████████████████████████████████████████████████████████

  STEP 2 | Computing ELO …
          5,690 (season,team) ELO entries
          369 teams with end-of-2025 ELO
  FIX ② | Applying conference strength penalty …
          3,766 team-seasons penalised
  STEP 3 | Building team stats, recent form, seeds …
          Team-season rows : 5,639
          Massey           : available
  STEP 4 | Building matchup feature matrix …
          Train rows : 85,809  |  Features : 172

  STEP 5 | Training LightGBM on 85,809 rows …
          ✓ LightGBM done
          Training XGBoost …
          ✓ XGBoost do

In [2]:
import subprocess
subprocess.run(["open", "-R", str(OUTPUT)])  # opens Finder and highlights the file

CompletedProcess(args=['open', '-R', '/Users/shaurya/Downloads/submission_stage2.csv'], returncode=0)

In [4]:
OUTPUT = DATA_DIR / "Updated_submission_stage2_final_one.csv"
```

Then **re-run the entire notebook from top to bottom** (Kernel → Restart & Run All).

After it finishes you should see:
```
File : /Users/shaurya/Downloads/Updated_submission_stage2_final_one.csv
✅ Updated_submission_stage2_final_one.csv ready — upload to Kaggle!
FIX ③ Seed overrides: 1 vs 16 → 97.5% min for 1-seed

SyntaxError: invalid character '→' (U+2192) (2713063671.py, line 4)

In [5]:
Michigan vs Howard

SyntaxError: invalid syntax (3646327750.py, line 1)